## Semana 2 Dia 2

Nosso primeiro projeto com um framework agentic!!

Prepare-se para algo ridiculamente fácil.

Vamos construir um sistema simples de Agentes para gerar e-mails de prospecção fria:
1. Fluxo de trabalho de Agente
2. Uso de ferramentas para chamar funções
3. Colaboração entre Agentes via Ferramentas e Handoffs

## Antes de começar - alguma configuração:


Por favor, visite o SendGrid em: https://sendgrid.com/

(SendGrid é uma empresa da Twilio para envio de e-mails.)

Crie uma conta — é grátis! (pelo menos, para mim, agora).

Depois de criar a conta, clique em:

Settings (barra lateral esquerda) >> API Keys >> Create API Key (botão no canto superior direito)

Copie a chave para a área de transferência e adicione uma nova linha ao seu arquivo .env:

`SENDGRID_API_KEY=xxxx`

E também, no SendGrid, vá para:

Settings (barra lateral esquerda) >> Sender Authentication >> "Verify a Single Sender"  
e verifique que o seu próprio endereço de e-mail é válido, para que o SendGrid possa enviar e-mails por você.

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio



In [2]:
load_dotenv(override=True)

True

## Etapa 1: Fluxo de trabalho do Agente

In [3]:
instructions1 = "Você é um agente de vendas que trabalha para a ComplAI, \
uma empresa que fornece uma ferramenta SaaS para garantir a conformidade SOC2 e preparar auditorias, com tecnologia de IA. \
Você escreve e-mails frios profissionais e sérios."

instructions2 = "Você é um agente de vendas bem-humorado e envolvente que trabalha para a ComplAI, \
uma empresa que fornece uma ferramenta SaaS para garantir a conformidade SOC2 e preparar auditorias, com tecnologia de IA. \
Você escreve e-mails frios espirituosos e envolventes, com grande chance de resposta."

instructions3 = "Você é um agente de vendas ocupado que trabalha para a ComplAI, \
uma empresa que fornece uma ferramenta SaaS para garantir a conformidade SOC2 e preparar auditorias, com tecnologia de IA. \
Você escreve e-mails frios concisos e diretos ao ponto."

In [4]:
sales_agent1 = Agent(
        name="Agente de Vendas Profissional",
        instructions=instructions1,
        model="gpt-5-mini"
)

sales_agent2 = Agent(
        name="Agente de Vendas Envolvente",
        instructions=instructions2,
        model="gpt-5-mini"
)

sales_agent3 = Agent(
        name="Agente de Vendas Ocupado",
        instructions=instructions3,
        model="gpt-5-mini"
)

In [5]:

result = Runner.run_streamed(sales_agent1, input="Escreva um e-mail de prospecção fria")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Assunto: Reduza tempo e riscos na preparação para auditoria SOC 2 — 15 minutos?

Olá [Nome],

Sou [Seu Nome], da ComplAI. Nós ajudamos equipes de segurança e compliance a tornar a conformidade SOC 2 mais simples, confiável e escalável com uma plataforma SaaS que usa IA para automatizar coleta de evidências, monitoramento contínuo e preparar relatórios prontos para auditoria.

Alguns benefícios típicos:
- Automação da coleta e organização de evidências para reduzir trabalho manual.
- Monitoramento contínuo de controles e alertas proativos para diminuir riscos entre auditorias.
- Relatórios e pacotes prontos para auditoria que aceleram o processo com seu auditor.
- Implementação rápida e integração com ferramentas comuns (ex.: cloud, IAM, tickets).

Se for relevante, posso:
- Fazer uma demonstração de 15 minutos focada nas suas principais dores,
- Enviar uma avaliação inicial sobre onde a ComplAI poderia reduzir esforço e exposição.

Qual o melhor horário nos próximos dias para conversar

In [6]:
message = "Escreva um e-mail de vendas frio"

with trace("E-mails frios em paralelo"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Assunto: Reduza o esforço para obter/renovar SOC 2 — 15 minutos?

Prezado(a) [Nome],

Sou [Seu Nome], da ComplAI. Notei que muitas empresas de tecnologia enfrentam processos manuais, falta de rastreabilidade e alto custo operacional ao preparar ou renovar a certificação SOC 2 — problemas que atrasam auditorias e elevam riscos.

A ComplAI é uma plataforma SaaS com inteligência artificial que automatiza a preparação para SOC 2: mapeamento automático de controles, coleta e organização de evidências, monitoramento contínuo e relatórios de prontidão para auditoria. O resultado é menos trabalho manual, menor probabilidade de não conformidades e uma preparação muito mais previsível e auditável.

Se for útil, proponho uma conversa rápida de 15 minutos para entender seu atual processo de conformidade e mostrar como ajudamos times de TI e segurança a acelerar auditorias e reduzir retrabalho. Tem disponibilidade esta semana? Posso enviar convite com alguns horários.

Atenciosamente,
[Seu Nome]
Ac

In [7]:
sales_picker = Agent(
    name="sales_picker",
    instructions="Você escolhe o melhor e-mail de prospecção fria entre as opções fornecidas. \
Imagine que você é um cliente e escolha aquele ao qual você tem mais probabilidade de responder. \
Não dê explicação; responda apenas com o e-mail selecionado.",
    model="gpt-5-mini"
)

In [8]:
message = "Escreva um e-mail de prospecção fria"

with trace("Seleção de vendedores"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "E-mails frios:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Melhor e-mail de vendas:\n{best.final_output}")


Melhor e-mail de vendas:
Assunto: SOC 2 sem dor de cabeça? (com uma pitada de IA)

Olá {{Nome}},

Se auditor fosse esporte, boa parte das empresas estaria no revezamento — sempre correndo atrás das evidências no último minuto. 😅

Sou [Seu Nome] da ComplAI. A gente ajuda times de segurança e compliance a trocar o sufoco por processo: nossa plataforma usa IA para automatizar coleta e organização de evidências, monitorar controles continuamente e gerar relatórios prontos para auditoria — tudo pensado para preparar (e passar) a avaliação SOC2 sem dramas de última hora.

O que isso costuma resolver na prática:
- Menos trabalho manual e menos “ops, cadê aquele arquivo?”
- Lacunas de conformidade identificadas rapidamente
- Relatórios e trilhas de auditoria centralizados e auditáveis

Topa uma demonstração rápida de 15 minutos? Posso mostrar um cenário real e como reduzir o esforço de preparação. Que tal terça ou quinta às 10h, ou prefere outro horário?

Se for mais fácil, responde com “mostr

Agora vá conferir o trace:

https://platform.openai.com/traces

## Parte 2: uso de ferramentas

Agora vamos adicionar uma ferramenta à mistura.

Lembra de todo aquele boilerplate JSON e da função `handle_tool_calls()` com a lógica de if..

In [9]:
sales_agent1 = Agent(
        name="Agente de Vendas Profissional",
        instructions=instructions1,
        model="gpt-5-mini",
)

sales_agent2 = Agent(
        name="Agente de Vendas Envolvente",
        instructions=instructions2,
        model="gpt-5-mini",
)

sales_agent3 = Agent(
        name="Agente de Vendas Ocupado",
        instructions=instructions3,
        model="gpt-5-mini",
)

In [10]:
sales_agent1

Agent(name='Agente de Vendas Profissional', instructions='Você é um agente de vendas que trabalha para a ComplAI, uma empresa que fornece uma ferramenta SaaS para garantir a conformidade SOC2 e preparar auditorias, com tecnologia de IA. Você escreve e-mails frios profissionais e sérios.', prompt=None, handoff_description=None, handoffs=[], model='gpt-5-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Etapas 2 e 3: Ferramentas e interações entre Agentes

Lembra de todo aquele boilerplate JSON?

Basta envolver sua função com o decorador `@function_tool`

In [11]:
@function_tool
def send_email(body: str):
    """ Envie um e-mail com o corpo fornecido para todos os prospects de vendas """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contato@santsoft.com")  # Altere para o seu remetente verificado
    to_email = To("santclear@gmail.com")  # Altere para o seu destinatário
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### Isso foi automaticamente convertido em uma ferramenta, com o boilerplate JSON criado

In [12]:
# Vamos dar uma olhada
send_email

FunctionTool(name='send_email', description='Envie um e-mail com o corpo fornecido para todos os prospects de vendas', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC6F93060>, strict_json_schema=True, is_enabled=True)

### E você também pode converter um Agente em uma ferramenta

In [13]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Escreva um e-mail de prospecção fria")
tool1

FunctionTool(name='sales_agent1', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC713C360>, strict_json_schema=True, is_enabled=True)

### Então agora podemos reunir todas as ferramentas:

Uma ferramenta para cada um de nossos 3 agentes que escrevem e-mails

E uma ferramenta para nossa função de enviar e-mails

In [14]:
description = "Escreva um e-mail de prospecção fria"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC6AF4720>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC4D67420>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input':

## E agora é a vez do nosso Gerente de Vendas — nosso agente de planejamento

In [15]:
instructions ="Você é um gerente de vendas que trabalha para a ComplAI. Você usa as ferramentas fornecidas para gerar e-mails de prospecção fria. \
Você nunca gera e-mails de vendas por conta própria; você sempre usa as ferramentas. \
Você experimenta as 3 ferramentas sales_agent uma vez antes de escolher a melhor. \
Você escolhe o único melhor e-mail e usa a ferramenta send_email para enviar o melhor e-mail (e apenas o melhor e-mail) ao usuário."


sales_manager = Agent(name="Gerente de Vendas", instructions=instructions, tools=tools, model="gpt-5-mini")

message = "Envie um e-mail de prospecção fria endereçado a 'Prezado CEO'"

with trace("Gerente de vendas"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Espere — você não recebeu um e-mail??</h2>
            <span style="color:#ff7800;">Com muitos agradecimentos ao aluno Chris S. por descrever seu problema e soluções. 
            Se você não receber um e-mail após executar a célula anterior, aqui estão algumas coisas para verificar: <br/>
            Primeiro, verifique sua pasta de Spam! Vários alunos não perceberam que os e-mails chegaram no Spam!<br/>Segundo, execute print(result) e veja se você está recebendo erros de SSL. 
            Se você estiver recebendo erros de SSL, confira estas <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">dicas de rede</a> e veja a nota na próxima célula. Também olhe o trace na OpenAI, e investigue no site do SendGrid, para buscar pistas. Avise-me se eu puder ajudar!
            </span>
        </td>
    </tr>
</table>

### E mais uma sugestão para enviar e-mails do aluno Oleksandr no Windows 11:

Se você estiver recebendo erros de certificado SSL, então:  
Execute isto em um terminal: `uv pip install --upgrade certifi`

Em seguida, execute este código:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Obrigado, Oleksandr!

## Lembre-se de verificar o trace

https://platform.openai.com/traces

E depois verifique seu e-mail!!

### Handoffs representam uma forma de um agente delegar para outro agente, passando o controle

Handoffs e Agentes-como-ferramentas são semelhantes:

Em ambos os casos, um Agente pode colaborar com outro Agente

Com ferramentas, o controle retorna

Com handoffs, o controle é transferido para o outro



In [16]:

subject_instructions = "Você pode escrever um assunto para um e-mail de prospecção fria. \
Você recebe uma mensagem e precisa escrever um assunto para um e-mail que provavelmente terá resposta."

html_instructions = "Você pode converter um corpo de e-mail em texto para um corpo de e-mail em HTML. \
Você recebe um corpo de e-mail em texto que pode conter algum markdown \
e precisa convertê-lo em um corpo de e-mail HTML com layout e design simples, claros e convincentes."

subject_writer = Agent(name="Redator de assunto de e-mail", instructions=subject_instructions, model="gpt-5-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Escreva um assunto para um e-mail de prospecção fria")

html_converter = Agent(name="Conversor de corpo de e-mail HTML", instructions=html_instructions, model="gpt-5-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Converter um corpo de e-mail em texto para um corpo de e-mail em HTML")


In [17]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envie um e-mail com o assunto informado e o corpo HTML para todos os prospects de vendas """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contato@santsoft.com")  # Altere para o seu remetente verificado
    to_email = To("santclear@gmail.com")  # Altere para o seu destinatário
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [18]:
tools = [subject_tool, html_tool, send_html_email]

In [19]:
tools

[FunctionTool(name='subject_writer', description='Escreva um assunto para um e-mail de prospecção fria', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC713CC20>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Converter um corpo de e-mail em texto para um corpo de e-mail em HTML', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC402FB00>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_html_email', description='Envie um e-mail com o 

In [20]:
instructions ="Você é um formatador e remetente de e-mails. Você recebe o corpo de um e-mail a ser enviado. \
Primeiro, você usa a ferramenta subject_writer para escrever um assunto para o e-mail; depois, usa a ferramenta html_converter para converter o corpo para HTML. \
Por fim, você usa a ferramenta send_html_email para enviar o e-mail com o assunto e o corpo em HTML."


emailer_agent = Agent(
    name="Gerente de E-mail",
    instructions=instructions,
    tools=tools,
    model="gpt-5-mini",
    handoff_description="Converter um e-mail para HTML e enviá-lo")


### Agora temos 3 ferramentas e 1 handoff

In [21]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC6AF4720>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001CBC4D67420>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Escreva um e-mail de prospecção fria', params_json_schema={'properties': {'input': {

In [ ]:
sales_manager_instructions = "Você é um gerente de vendas da ComplAI. Você usa as ferramentas que lhe foram disponibilizadas para gerar e-mails de vendas frios. \
Você nunca gera e-mails de vendas por conta própria; você sempre usa as ferramentas. \
Você experimenta as 3 ferramentas de agente de vendas pelo menos uma vez antes de escolher a melhor. \
Você pode usar as ferramentas várias vezes se não estiver satisfeito com os resultados da primeira tentativa. \
Você seleciona o único melhor e-mail usando seu próprio julgamento sobre qual e-mail será mais eficaz. \
Depois de escolher o e-mail, você faz handoff para o agente Gerente de E-mail para formatar e enviar o e-mail."


sales_manager = Agent(
    name="Gerente de Vendas",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-5-mini")

message = "Envie um e-mail de prospecção fria endereçado a 'Prezado CEO', de Alice"

with trace("SDR automatizado"):
    result = await Runner.run(sales_manager, message)

### Lembre-se de verificar o trace

https://platform.openai.com/traces

E depois verifique seu e-mail!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercício</h2>
            <span style="color:#ff7800;">Você consegue identificar os padrões de design agentic que foram usados aqui?<br/>
            Qual é a 1 linha que mudou isto de um “workflow” agentic para “agent” segundo a definição da Anthropic?<br/>
            Tente adicionar mais ferramentas e Agentes! Você pode ter ferramentas que façam mala direta para enviar para uma lista.<br/><br/>
            DESAFIO DIFÍCIL: pesquise como fazer o SendGrid chamar um webhook de Callback quando um usuário responder a um e-mail,
            Depois faça o SDR responder para manter a conversa! Isso pode exigir um pouco de “vibe coding” 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implicações comerciais</h2>
            <span style="color:#00bfff;">Isso é imediatamente aplicável à Automação de Vendas; mas, mais genericamente, pode ser aplicado à automação ponta a ponta de qualquer processo de negócio por meio de conversas e ferramentas. Pense em maneiras de aplicar uma solução de Agentes
            como esta no seu dia a dia.
            </span>
        </td>
    </tr>
</table>

## Nota extra:

O Google anunciou o Agent Development Kit (ADK), que está em early preview. Ainda está em desenvolvimento, então é cedo para cobrirmos aqui. Mas é interessante notar que ele parece bastante semelhante ao OpenAI Agents SDK. Para dar um gostinho, aqui vai um exemplo de código do ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Bem, isso parece familiar!